In [1]:
# draw number

# Import numpy for arrays and matplotlib for drawing the numbers
import numpy as np
import matplotlib.pyplot as plt

def draw_number(image_num):
    # Open the 100 training samples in read mode
    # data_file = open('mnist_train_100.csv','r')
    # data_file = open('mnist_train.csv','r')
    # data_file = open('fashion_mnist_train_100.csv','r')
    # data_file = open('fashion_mnist_train.csv','r')
    # data_file = open('mnist_test_100.csv','r')
    # data_file = open('mnist_test.csv','r')
    # data_file = open('fashion_mnist_test_100.csv','r')
    data_file = open('fashion_mnist_test.csv','r')

    # Read all of the lines from the file into memory
    data_list = data_file.readlines()

    # Close the file (we are done with it)
    data_file.close()

    # Take the first line (data_list index 0, the first sample), and split it up based on the commas
    # all_values now contains a list of [label, pixel 1, pixel 2, pixel 3, ... , pixel 784]
    all_values = data_list[image_num].split(',')

    # Take the long list of pixels (but not the label), and reshape them to a 2D array of pixels
    image_array = np.asfarray(all_values[1:]).reshape((28, 28))

    # PLot this 2D array as an image, use the grey colour map and don't interpolate
    plt.imshow(image_array, cmap='Greys', interpolation='None')
    plt.show()


In [2]:
# import scipy.special for the sigmoid function expit()
import scipy.special, numpy

# Neural network class definition
class NeuralNetwork:
    # Init the network, this gets run whenever we make a new instance of this class
    def __init__(self, input_nodes, hidden_nodes_1, hidden_nodes_2, output_nodes, learning_rate):
        # Set the number of nodes in each input, hidden and output layer
        self.i_nodes = input_nodes
        self.h1_nodes = hidden_nodes_1
        self.h2_nodes = hidden_nodes_2
        self.o_nodes = output_nodes

        # Weight matrices, with (input -> hidden) and who (hidden -> output)
        self.wih = numpy.random.normal(0.0, pow(self.h1_nodes, -0.5), (self.h1_nodes, self.i_nodes))
        self.whh = numpy.random.normal(0.0, pow(self.h2_nodes, -0.5), (self.h2_nodes, self.h1_nodes))
        self.who = numpy.random.normal(0.0, pow(self.o_nodes, -0.5), (self.o_nodes, self.h2_nodes))

        # Set the learning rate
        self.lr = learning_rate

        # Set the activation function, the logistic sigmoid
        self.activation_function = lambda x: scipy.special.expit(x)

    # Train the network using back-propagation of errors
    def train(self, inputs_list, targets_list):
        # Convert inputs into 2D arrays
        inputs_array = numpy.array(inputs_list, ndmin=2).T
        targets_array = numpy.array(targets_list, ndmin=2).T

        # Calculate signals into hidden layer
        hidden_inputs1 = numpy.dot(self.wih, inputs_array)

        # Calculate signals emerging from hidden layer
        hidden_outputs1 = self.activation_function(hidden_inputs1)

        # Calculate signals into hidden layer
        hidden_inputs2 = numpy.dot(self.whh, hidden_outputs1)

        # Calculate signals emerging from hidden layer
        hidden_outputs2 = self.activation_function(hidden_inputs2)

        # Calculate signals into final output layer
        final_inputs = numpy.dot(self.who, hidden_outputs2)

        # Calculate the signals emerging from final output layer
        final_outputs = self.activation_function(final_inputs)

        # Current error is (target - actual)
        output_errors = targets_array - final_outputs

        # Hidden layer errors are the output errors, split by the weights, recombined at hidden nodes
        hidden_errors1 = numpy.dot(self.who.T, output_errors)

        # Hidden layer errors are the output errors, split by the weights, recombined at hidden nodes
        hidden_errors2 = numpy.dot(self.whh.T, hidden_errors1)

        # Update the weights for the links between the hidden and output layers
        self.who += self.lr*numpy.dot((output_errors*final_outputs*(1.0 - final_outputs)), numpy.transpose(hidden_outputs2))
        # Update the weights for the links between the input and hidden layers
        self.whh += self.lr*numpy.dot((hidden_errors1*hidden_outputs2*(1.0 - hidden_outputs2)), numpy.transpose(hidden_outputs1))
        # Update the weights for the links between the input and hidden layers
        self.wih += self.lr*numpy.dot((hidden_errors2*hidden_outputs1*(1.0 - hidden_outputs1)), numpy.transpose(inputs_array))

    # Query the network
    def query(self, inputs_list):
        # Converts the inputs list into a 2D array
        inputs_array = numpy.array(inputs_list, ndmin=2).T

        # Calcualte the signals into hidden layer
        hidden_inputs1 = numpy.dot(self.wih, inputs_array) 

        # Calculate output from the hidden layer
        hidden_outputs1 = self.activation_function(hidden_inputs1)

        # Calcualte the signals into hidden layer
        hidden_inputs2 = numpy.dot(self.whh, hidden_outputs1) 

        # Calculate output from the hidden layer
        hidden_outputs2 = self.activation_function(hidden_inputs2)

        # Calculate signals into final layer
        final_inputs = numpy.dot(self.who, hidden_outputs2)

        # Calculate outputs from the final layer
        final_outputs = self.activation_function(final_inputs)
        print(final_outputs)
        return final_outputs


In [3]:

# parameters

# Exercise 1
# input_nodes = 784
# hidden_nodes = 100
# output_nodes = 10
# learning_rate = 0.3

# Exercise 4
input_nodes = 784
hidden_nodes1 = 128 #100
hidden_nodes2 = 64 #100
output_nodes = 10 
learning_rate = 0.125 #0.3

n = NeuralNetwork(input_nodes, hidden_nodes1, hidden_nodes2, output_nodes, learning_rate)


In [4]:
# training

# Load the MNIST 100 training samples CSV file into a list
# training_data_file = open("mnist_train_100.csv",'r')
training_data_file = open("mnist_train.csv",'r')
# training_data_file = open("fashion_mnist_train_100.csv",'r')
# training_data_file = open("fashion_mnist_train.csv",'r')
training_data_list = training_data_file.readlines()
training_data_file.close()

# Train the neural network on each training sample
for record in training_data_list:
    # Split the record by the commas
    all_values = record.split(',')
    # Scale and shift the inputs from 0..255 to 0.01..1
    inputs = (np.asfarray(all_values[1:])/255.0*0.99) + 0.01
    # Create the target output values (all 0.01, except the desired label which is 0.99)
    targets = np.zeros(output_nodes) + 0.01
    # All_values[0] is the target label for this record
    targets[int(all_values[0])] = 0.99
    # Train the network
    for i in range(40):
        n.train(inputs,targets)
    # n.train(inputs,targets)
pass


In [5]:
# testing

# Load the MNIST test samples CSV file into a list
# test_data_file = open("mnist_test_10.csv",'r')
test_data_file = open("mnist_test.csv",'r')
# test_data_file = open("fashion_mnist_test_10.csv",'r')
# test_data_file = open("fashion_mnist_test.csv",'r')
test_data_list = test_data_file.readlines()
test_data_file.close()

# Scorecard list for how well the network performs, initially empty
scorecard = []

# Loop through all of the records in the test data set
for record in test_data_list:
    # Split the record by the commas
    all_values = record.split(',')
    # The correct label is the first value
    correct_label = int(all_values[0])
    print(correct_label, "Correct label")
    # Scale and shift the inputs
    inputs = (np.asfarray(all_values[1:])/255.0*0.99)+0.01
    # Query the network
    outputs = n.query(inputs)
    # The index of the highest value output corresponds to the label
    label = np.argmax(outputs)
    print(label,"Network label")
    # Append either a 1 or a 0 to the scorecard list
    if (label == correct_label):
        scorecard.append(1)
    else:
        scorecard.append(0)
        pass
    pass

# Calculate the performance score, the fraction of correct answers
scorecard_array = np.asarray(scorecard)


7 Correct label
[[3.42602819e-03]
 [3.35889461e-03]
 [5.41279150e-03]
 [1.07290758e-02]
 [4.73629628e-02]
 [2.74359608e-03]
 [4.69311464e-03]
 [9.77821421e-01]
 [5.56072120e-04]
 [1.02699013e-02]]
7 Network label
2 Correct label
[[0.05932782]
 [0.08011631]
 [0.25340926]
 [0.03777443]
 [0.04823582]
 [0.05365476]
 [0.58757342]
 [0.07294687]
 [0.80493384]
 [0.025602  ]]
8 Network label
1 Correct label
[[2.39458513e-03]
 [8.77456948e-01]
 [1.59587523e-03]
 [9.10037034e-04]
 [1.42267806e-02]
 [3.99672152e-04]
 [5.63719528e-02]
 [7.46596969e-03]
 [1.24362013e-02]
 [3.84647861e-04]]
1 Network label
0 Correct label
[[0.03465224]
 [0.05392241]
 [0.09144553]
 [0.03027575]
 [0.03566857]
 [0.0320546 ]
 [0.33175753]
 [0.04957375]
 [0.93795886]
 [0.04744775]]
8 Network label
4 Correct label
[[1.93395577e-04]
 [1.43918452e-04]
 [4.44675406e-04]
 [9.84423825e-05]
 [5.97999204e-02]
 [7.97773145e-02]
 [1.59972310e-03]
 [3.68114885e-03]
 [6.87302853e-01]
 [8.20962243e-03]]
8 Network label
1 Correct label

In [6]:
print("Performance = ", (scorecard_array.sum()/scorecard_array.size)*100, '%')

Performance =  48.54 %


In [7]:
# Exercise 3: Display images (I think I could have got these classifications...)
# for i in range(len(scorecard_array)):
#     if scorecard_array[i] == 0:
#         draw_number(i)
